# 3. Model Training — NSL-KDD Intrusion Detection

**Goal:** train 3 different classifiers and compare them — not just to pick a winner, but because comparing reveals real insight about which model actually generalizes to new attacks.

**Input:** `train_processed.csv`, `test_processed.csv` (from notebook 2)

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

train = pd.read_csv("data/train_processed.csv")
test = pd.read_csv("data/test_processed.csv")

print(train.shape, test.shape)

(125973, 43) (22544, 44)


In [2]:
feature_cols = [c for c in train.columns if c not in ["label", "attack_category"]]

X_train = train[feature_cols]
y_train = train["attack_category"]
X_test = test[[c for c in feature_cols if c != "is_novel_attack_type"]]
y_test = test["attack_category"]
is_novel = test["is_novel_attack_type"]

print("Features used:", len(X_test.columns))

Features used: 41


## Scale features for Logistic Regression

Logistic Regression is sensitive to feature scale (a feature ranging 0-1000 can dominate one ranging 0-1 even if it's not more important). Tree-based models (Decision Tree, Random Forest) don't need this — they split on thresholds, not magnitudes.

In [3]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Train three models

**Why `class_weight="balanced"` matters:** without it, a model could predict "normal" for everything and still score ~55% accuracy, because normal traffic dominates the dataset. This setting forces the model to pay attention to rare classes like U2R (only 52 training examples).

In [4]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", max_depth=15, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                             max_depth=20, random_state=42, n_jobs=-1),
}

fitted_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
    else:
        model.fit(X_train, y_train)
    fitted_models[name] = model

print("Done.")

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Done.


## Evaluate each model

We use **macro F1** as the main metric, not accuracy. Accuracy gets fooled by class imbalance (predicting "normal" a lot looks good); macro F1 treats every class — even tiny U2R — as equally important to get right.

In [5]:
results = {}

for name, model in fitted_models.items():
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    if name == "Logistic Regression":
        y_pred = model.predict(X_test_scaled)
    else:
        y_pred = model.predict(X_test)

    print(classification_report(y_test, y_pred, zero_division=0))

    macro_f1 = f1_score(y_test, y_pred, average="macro")
    weighted_f1 = f1_score(y_test, y_pred, average="weighted")

    results[name] = {"y_pred": y_pred, "macro_f1": macro_f1, "weighted_f1": weighted_f1}
    print(f"Macro F1: {macro_f1:.3f} | Weighted F1: {weighted_f1:.3f}")


Logistic Regression
              precision    recall  f1-score   support

         DoS       0.98      0.78      0.87      7460
       Probe       0.75      0.88      0.81      2421
         R2L       0.87      0.26      0.40      2885
         U2R       0.08      0.57      0.14        67
      normal       0.74      0.94      0.83      9711

    accuracy                           0.79     22544
   macro avg       0.68      0.69      0.61     22544
weighted avg       0.83      0.79      0.78     22544

Macro F1: 0.609 | Weighted F1: 0.782

Decision Tree
              precision    recall  f1-score   support

         DoS       0.96      0.79      0.87      7460
       Probe       0.66      0.71      0.68      2421
         R2L       0.20      0.01      0.02      2885
         U2R       0.42      0.07      0.13        67
      normal       0.69      0.97      0.81      9711

    accuracy                           0.76     22544
   macro avg       0.58      0.51      0.50     22544
weig

## The key test: how well does each model handle NOVEL attacks?

This is the most important evaluation in the whole project. We already flagged which test rows are attack types the model never saw during training. Now let's check: does the model at least recognize them as "not normal," even without knowing exactly what they are?

In [6]:
novel_mask = (is_novel == 1).values

print("Novel-attack detection rate by model:")
print("(% of never-seen attack types correctly flagged as NOT normal)\n")

for name, r in results.items():
    y_pred = r["y_pred"]
    detected = (y_pred[novel_mask] != "normal").mean()
    results[name]["novel_detection_rate"] = detected
    print(f"{name}: {detected:.1%}")

Novel-attack detection rate by model:
(% of never-seen attack types correctly flagged as NOT normal)

Logistic Regression: 43.5%
Decision Tree: 42.8%
Random Forest: 15.1%


**Expected pattern:** Random Forest usually looks strong on standard metrics but often has the *worst* novel-attack detection rate — a sign it's overfitting to the specific attack signatures it memorized during training, rather than learning generalizable "this looks suspicious" patterns. Logistic Regression, despite being the simplest model, tends to generalize better here.

If your results show this same pattern, that's a genuinely interesting, real finding worth highlighting — not just "I ran three models," but "I found evidence that model complexity can hurt generalization to novel attacks."

## Pick the best model and look at its confusion matrix

In [7]:
best_name = max(results, key=lambda k: results[k]["macro_f1"])
print(f"Best model by macro F1: {best_name}")

y_pred_best = results[best_name]["y_pred"]
labels_order = ["normal", "DoS", "Probe", "R2L", "U2R"]
cm = confusion_matrix(y_test, y_pred_best, labels=labels_order)
cm_df = pd.DataFrame(cm, index=[f"actual_{l}" for l in labels_order],
                      columns=[f"pred_{l}" for l in labels_order])
cm_df

Best model by macro F1: Logistic Regression


,pred_normal,pred_DoS,pred_Probe,pred_R2L,pred_U2R
actual_normal,9161,58,274,99,119
actual_DoS,1346,5836,272,0,6
actual_Probe,209,81,2128,2,1
actual_R2L,1689,0,161,740,295
actual_U2R,14,0,2,13,38


**How to read this:** rows are the true label, columns are what the model predicted. The diagonal is correct predictions. Look at the `actual_R2L` and `actual_U2R` rows — you'll likely see a lot of them misclassified as `normal`, which is the known weak spot of this dataset (too few training examples for those attack types).

## Save results for the next notebook

In [8]:
summary_rows = []
for name, r in results.items():
    summary_rows.append({
        "model": name,
        "macro_f1": round(r["macro_f1"], 4),
        "weighted_f1": round(r["weighted_f1"], 4),
        "novel_attack_detection_rate": round(r["novel_detection_rate"], 4),
    })
model_comparison = pd.DataFrame(summary_rows)
model_comparison.to_csv("data/model_comparison.csv", index=False)
cm_df.to_csv("data/confusion_matrix.csv")

test_out = test[["label", "attack_category", "is_novel_attack_type"]].copy()
test_out["predicted_category"] = y_pred_best
test_out.to_csv("data/test_predictions.csv", index=False)

print(model_comparison)

                 model  macro_f1  weighted_f1  novel_attack_detection_rate
0  Logistic Regression    0.6093       0.7821                       0.4347
1        Decision Tree    0.5016       0.7121                       0.4280
2        Random Forest    0.4738       0.6906                       0.1507


## What's next

Open **`04_evaluation_explainability.ipynb`** to dig into feature importance (why does the model predict what it predicts?) and build final charts.